Attach compute cluster. Serverless cannot be used for this notebook.

In [ ]:
from datetime import date
from utils.storage_io import ADLSConfig, StorageIO

import utils.customer_master as customer_master
import utils.etf_master as etf_master
import utils.etf_trades as etf_trades
import utils.customer_etf_portfolio as customer_etf_portfolio
import utils.customer_cash_portfolio as customer_cash_portfolio

In [ ]:
dbutils.widgets.text("storage_account", "adlsworkshopadb")
dbutils.widgets.text("container", "landing")
dbutils.widgets.text("state_dir", "simulation")
dbutils.widgets.text("state_file", "simulation_state.json")

dbutils.widgets.text("default_start_date", "2026-01-01")
dbutils.widgets.text("default_end_date", "2026-02-28")

dbutils.widgets.text("customer_n", "500")
dbutils.widgets.text("seed", "42")

In [ ]:
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
state_dir = dbutils.widgets.get("state_dir")
state_file = dbutils.widgets.get("state_file")

default_start_date = dbutils.widgets.get("default_start_date")
default_end_date = dbutils.widgets.get("default_end_date")

krx_auth_key = dbutils.widgets.get("krx_auth_key")
customer_n = int(dbutils.widgets.get("customer_n"))
seed = int(dbutils.widgets.get("seed"))

In [ ]:
# Paths (Bronze Delta)

def abfss(container: str, storage: str, path: str) -> str:
    p = path.lstrip("/")
    return f"abfss://{container}@{storage}.dfs.core.windows.net/{p}"

paths = {
    "customer_master": abfss(container, storage_account, "data/customer_master"),
    "etf_master": abfss(container, storage_account, "data/etf_master"),
    "etf_trades": abfss(container, storage_account, "data/etf_trades"),
    "customer_etf_portfolio": abfss(container, storage_account, "data/customer_etf_portfolio"),
    "customer_cash_portfolio": abfss(container, storage_account, "data/customer_cash_portfolio"),
}

cfg = {
    "paths": paths,
    "krx_auth_key": krx_auth_key,
    "customer_n": customer_n,
    "seed": seed,
    # "source_sys": "krx_api" if krx_auth_key else "local",
    "partition_by_base_date": True,
    # optional fallback ETF list
    "etf_codes": [f"ETF{str(i).zfill(4)}" for i in range(1, 21)]
}

In [ ]:
# ADLS authentication via storage account key (like shared notebook pattern)
adls_access_key = dbutils.widgets.get("adls_access_key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    adls_access_key
)

# Load / init simulation state JSON in ADLS

state_cfg = ADLSConfig(storage_account=storage_account, container=container, base_dir=state_dir)
io = StorageIO(dbutils=dbutils, cfg=state_cfg)
state_path = state_cfg.abfss_path(state_file)

state = io.load_or_init_simulation_state(
    state_path=state_path,
    start_date=default_start_date,
    end_date=default_end_date
)

In [ ]:
# Spark accelerate

spark.conf.set("spark.sql.shuffle.partitions", "512")
spark.conf.set("spark.default.parallelism", "512")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

In [ ]:
from datetime import date, timedelta

# ----------------------------
# Load state
# ----------------------------
cur = date.fromisoformat(state["current_date"])
end = date.fromisoformat(state["end_date"])

new_end = date.fromisoformat(
    dbutils.widgets.get("default_end_date")
)

if new_end > end:
    end = new_end
    state["end_date"] = end.isoformat()

state["running"] = True
io.write_json_atomic(state_path, state, overwrite=True)

# ----------------------------
# Process all days from current_date to end_date
# ----------------------------
processed = []

while cur <= end:
    base_date = cur.isoformat()
    is_month_start = (cur.day == 1)

    # Monthly masters at month boundary
    if is_month_start:
        customer_master.run(base_date=base_date, cfg=cfg)
        etf_master.run(base_date=base_date, cfg=cfg)

    # Daily updates
    etf_trades.run(base_date=base_date, cfg=cfg)
    customer_cash_portfolio.run(base_date=base_date, cfg=cfg)
    customer_etf_portfolio.run(base_date=base_date, cfg=cfg)

    processed.append(base_date)
    print(f"Processed: {base_date} (monthly={is_month_start})")

    # Advance and persist state
    cur += timedelta(days=1)
    state["current_date"] = cur.isoformat()
    state["day_counter"] = int(state.get("day_counter", 0)) + 1
    io.write_json_atomic(state_path, state, overwrite=True)

# ----------------------------
# Mark simulation complete
# ----------------------------
state["running"] = False
io.write_json_atomic(state_path, state, overwrite=True)

dbutils.notebook.exit(
    f"OK: processed {len(processed)} days "
    f"({processed[0]} ~ {processed[-1]})"
)